In [ ]:
import pandas as pd

df = pd.read_csv("customer_booking.csv", encoding="latin=1")
df.head()

,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
0,2,Internet,RoundTrip,262,19,7,Sat,AKLDEL,New Zealand,1,0,0,5.52,0
1,1,Internet,RoundTrip,112,20,3,Sat,AKLDEL,New Zealand,0,0,0,5.52,0
2,2,Internet,RoundTrip,243,22,17,Wed,AKLDEL,India,1,1,0,5.52,0
3,1,Internet,RoundTrip,96,31,4,Sat,AKLDEL,New Zealand,0,0,1,5.52,0
4,2,Internet,RoundTrip,68,22,15,Wed,AKLDEL,India,1,0,1,5.52,0


In [ ]:
df.info()

df.columns
df.isnull()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_passengers         50000 non-null  int64  
 1   sales_channel          50000 non-null  object 
 2   trip_type              50000 non-null  object 
 3   purchase_lead          50000 non-null  int64  
 4   length_of_stay         50000 non-null  int64  
 5   flight_hour            50000 non-null  int64  
 6   flight_day             50000 non-null  object 
 7   route                  50000 non-null  object 
 8   booking_origin         50000 non-null  object 
 9   wants_extra_baggage    50000 non-null  int64  
 10  wants_preferred_seat   50000 non-null  int64  
 11  wants_in_flight_meals  50000 non-null  int64  
 12  flight_duration        50000 non-null  float64
 13  booking_complete       50000 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.3+ 

,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,False,False,False,False,False,False,False,False,False,False,False,False,False,False
49996,False,False,False,False,False,False,False,False,False,False,False,False,False,False
49997,False,False,False,False,False,False,False,False,False,False,False,False,False,False
49998,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [ ]:
df.shape

(50000, 14)

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("airline.db")

df = pd.read_csv("customer_booking.csv", encoding="latin=1")

df.to_sql("customer_booking", conn,  if_exists="replace", index=False)
print("Table created: Customer_booking")


Table created: Customer_booking


In [ ]:
query= """
select *
from customer_booking
limit 5;
"""
pd.read_sql(query, conn)

,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
0,2,Internet,RoundTrip,262,19,7,Sat,AKLDEL,New Zealand,1,0,0,5.52,0
1,1,Internet,RoundTrip,112,20,3,Sat,AKLDEL,New Zealand,0,0,0,5.52,0
2,2,Internet,RoundTrip,243,22,17,Wed,AKLDEL,India,1,1,0,5.52,0
3,1,Internet,RoundTrip,96,31,4,Sat,AKLDEL,New Zealand,0,0,1,5.52,0
4,2,Internet,RoundTrip,68,22,15,Wed,AKLDEL,India,1,0,1,5.52,0


In [ ]:
query = """
select count(*) as total_bookings
from customer_booking;
"""
pd.read_sql(query, conn)

,total_bookings
0,50000


In [ ]:
query = """
select
   booking_complete,
   count(*) as booking
from customer_booking
group by booking_complete;
"""
pd.read_sql(query, conn)

,booking_complete,booking
0,0,42522
1,1,7478


In [ ]:
query = """
select
   round(
    100.0 * sum(case when booking_complete = 1 then 1 else 0 end) / count(*),
    2
   ) as booking_completion_rate
from customer_booking;
"""
pd.read_sql(query, conn)

,booking_completion_rate
0,14.96


In [ ]:
query = """
select sales_channel,
       count(*) as total_bookings,
       sum(booking_complete) as completed_bookings
from customer_booking
group by sales_channel
order by completed_bookings desc;
"""
pd.read_sql(query, conn)


,sales_channel,total_bookings,completed_bookings
0,Internet,44382,6869
1,Mobile,5618,609


In [ ]:
query = """
select
    trip_type,
    count(*) as total_bookings,
    sum(booking_complete) as completed_bookings
from customer_booking
group by trip_type
"""
pd.read_sql(query, conn)

,trip_type,total_bookings,completed_bookings
0,CircleTrip,116,5
1,OneWay,387,20
2,RoundTrip,49497,7453


In [ ]:
query = """
select
    case
      when purchase_lead < 30 then '0-30 days'
      when purchase_lead between 30 and 90 then '30-90 days'
      else '90+ days'
   end as lead_bucket,
      count(*) as total_bookings,
      sum(booking_complete) as completed_bookings
from customer_booking
group by lead_bucket
order by lead_bucket
"""
pd.read_sql(query, conn)

,lead_bucket,total_bookings,completed_bookings
0,0-30 days,16721,2754
1,30-90 days,17433,2544
2,90+ days,15846,2180


In [ ]:
query = """
  select
     flight_day,
     count(*) as total_bookings,
     sum(booking_complete) as completed_bookings,
     round(
        100.0 * sum(case when booking_complete = 1 then 1 else 0 end) / count(*),
        3
     ) as completion_rate
  from customer_booking
  group by flight_day
  order by total_bookings
"""
pd.read_sql(query, conn)

,flight_day,total_bookings,completed_bookings,completion_rate
0,Sat,5812,861,14.814
1,Sun,6554,927,14.144
2,Fri,6761,983,14.539
3,Thu,7424,1122,15.113
4,Tue,7673,1129,14.714
5,Wed,7674,1252,16.315
6,Mon,8102,1204,14.861


In [ ]:
query = """
select
   route,
   count(*) as total_bookings
from customer_booking
group by route
order by total_bookings desc
limit 10;
"""
pd.read_sql(query, conn)

,route,total_bookings
0,AKLKUL,2680
1,PENTPE,924
2,MELSGN,842
3,ICNSIN,801
4,DMKKIX,744
5,ICNSYD,695
6,DMKPER,679
7,DPSICN,666
8,DMKOOL,655
9,MELPEN,649


In [ ]:
query = """
 select
    booking_origin,
    count(*) as total_bookings,
    sum(booking_complete) as completed_bookings
from customer_booking
group by booking_origin
order by total_bookings desc
limit 5;
"""
pd.read_sql(query, conn)

,booking_origin,total_bookings,completed_bookings
0,Australia,17872,900
1,Malaysia,7174,2468
2,South Korea,4559,462
3,Japan,3885,478
4,China,3387,694


In [ ]:
query = """
 select
  wants_extra_baggage,
  count(*) as total_bookings,
  sum(booking_complete) as completion,
  round(
    100.0 * sum(case when booking_complete = 1 then 1 else 0 end) / count(*),
    2
  ) as completion_rate
from customer_booking
group by wants_extra_baggage;
"""
pd.read_sql(query, conn)

,wants_extra_baggage,total_bookings,completion,completion_rate
0,0,16561,1905,11.50
1,1,33439,5573,16.67


In [ ]:
query = """
select
  wants_preferred_seat,
  wants_in_flight_meals,
  count(*) as total_bookings,
  sum(booking_complete) as completion,
  round(
    100.0 * sum(case when booking_complete = 1 then 1 else 0 end) / count(*),
    2
  ) as rate_successfull
from customer_booking
group by wants_preferred_seat, wants_in_flight_meals;
"""
pd.read_sql(query, conn)

,wants_preferred_seat,wants_in_flight_meals,total_bookings,completion,rate_successfull
0,0,0,23698,3226,13.61
1,0,1,11454,1623,14.17
2,1,0,4945,824,16.66
3,1,1,9903,1805,18.23


In [ ]:
query = """
 select
   round(avg(flight_duration), 2) as avg_flight_duration
 from customer_booking;
 """
pd.read_sql(query, conn)

,avg_flight_duration
0,7.28


In [ ]:
query = """
 select
   round(avg(length_of_stay), 1) as avg_length_of_stay
from customer_booking;
"""
pd.read_sql(query, conn)

,avg_length_of_stay
0,23.0


In [ ]:
query = """
select
  sales_channel,
  round(100.0 * sum(booking_complete) / count(*), 2) as conversion_rate
from customer_booking
group by sales_channel;
"""
pd.read_sql(query, conn)

,sales_channel,conversion_rate
0,Internet,15.48
1,Mobile,10.84


In [ ]:
query = """
  select
   route,
   count(*) as total_bookings,
   round(100.0 * sum(booking_complete) / count(*), 5) as conversion_rate
  from customer_booking
  group by route
  having total_bookings > 500
  order by conversion_rate desc;
"""
pd.read_sql(query, conn)

,route,total_bookings,conversion_rate
0,PENTPE,924,43.39827
1,DMKKIX,744,25.13441
2,AKLKUL,2680,21.15672
3,MELPEN,649,21.10940
4,ICNSIN,801,11.23596
5,SGNSYD,614,10.09772
6,DPSICN,666,8.55856
7,COKSYD,511,6.84932
8,DMKSYD,532,6.76692
9,DMKOOL,655,6.10687


In [ ]:
query = """
 select
   wants_extra_baggage,
   wants_in_flight_meals,
   wants_preferred_seat,
   round( 100.0 * sum(booking_complete) / count(*), 2) as conversion_rate
 from customer_booking
 group by wants_extra_baggage, wants_in_flight_meals, wants_preferred_seat
 order by conversion_rate desc;
 """
pd.read_sql(query, conn)


,wants_extra_baggage,wants_in_flight_meals,wants_preferred_seat,conversion_rate
0,1,1,1,18.60
1,1,0,1,18.33
2,1,0,0,15.93
3,0,1,1,15.36
4,1,1,0,15.06
5,0,0,1,13.02
6,0,1,0,12.07
7,0,0,0,10.67


In [ ]:
query = """
 select
